# LangGraph Agent — Sample Code (template)
### Module 6 sample notebook

A minimal LangGraph agent you can copy: state → node → conditional edge → node → END. Pure structure (no API key needed). See `langgraph_part1.ipynb` / `langgraph_part2.ipynb` for the full lessons. `pip install langgraph`.

In [ ]:
# pip install langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1) state = shared memory
class State(TypedDict):
    text: str
    label: str
    result: str

# 2) nodes = functions that read/update state
def classify(s: State) -> dict:
    return {"label": "urgent" if "critical" in s["text"].lower() else "normal"}

def handle_urgent(s: State) -> dict:
    return {"result": "Escalated: " + s["text"]}

def handle_normal(s: State) -> dict:
    return {"result": "Logged: " + s["text"]}

# 3) router for the conditional edge
def route(s: State) -> str:
    return s["label"]

# 4) build the graph
b = StateGraph(State)
b.add_node("classify", classify)
b.add_node("urgent", handle_urgent)
b.add_node("normal", handle_normal)
b.set_entry_point("classify")
b.add_conditional_edges("classify", route, {"urgent": "urgent", "normal": "normal"})
b.add_edge("urgent", END)
b.add_edge("normal", END)
graph = b.compile()

# 5) run it
for t in ["Critical RCE on web server", "Routine config change"]:
    out = graph.invoke({"text": t, "label": "", "result": ""})
    print(t, "->", out["result"])